In [1]:
import tensorflow as tf
from pathlib import Path

In [14]:
tf.random.set_seed(123)

project_dir = Path.cwd().parent
train_dir = project_dir / "img/train"
validation_dir = project_dir / "img/validate"

img_height = 180
img_width = 180
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

""" 
I created another notebook to give EfficientNetB0 architecture a try but there seems 
to be an issue with my versions of tf and keras as I tried many different tweaks and 
kept getting a shape mismatch error, so the MobileNetV2 architecture below seems 
like the best fit for my small dataset if I want to use transfer learning
"""
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

@tf.keras.utils.register_keras_serializable()
class MobileNetV2Preprocess(tf.keras.layers.Layer):
    def call(self, x):
        return tf.keras.applications.mobilenet_v2.preprocess_input(x)

model = tf.keras.Sequential([
    MobileNetV2Preprocess(),
    tf.keras.layers.RandomFlip('horizontal'),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

model.save('../models/mobilenetv2_model.keras')

Found 161 files belonging to 3 classes.


Found 42 files belonging to 3 classes.


/var/folders/04/s_rk8gfn6vq1608hxwwtl_040000gn/T/ipykernel_16093/3975237494.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 6s 584ms/step - accuracy: 0.3416 - loss: 1.6511 - val_accuracy: 0.5476 - val_loss: 0.9689
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 119ms/step - accuracy: 0.5404 - loss: 1.3506 - val_accuracy: 0.7381 - val_loss: 0.6390
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.5839 - loss: 1.0923 - val_accuracy: 0.7381 - val_loss: 0.6940
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.6460 - loss: 0.9486 - val_accuracy: 0.8333 - val_loss: 0.4071
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.6957 - loss: 0.9798 - val_accuracy: 0.8333 - val_loss: 0.3653
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - accuracy: 0.6832 - loss: 0.8599 - val_accuracy: 0.8095 - val_loss: 0.3781
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 91ms/step - accuracy: 0.6335 - loss: 0.8804 - val_accuracy: 0.8333 - val_loss: 0.3707
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - accuracy: 0.7702 - loss: 0.5994 - val_accuracy: 0.7857 - val_loss: 0.4